In [ ]:
#author: niels schmidt martinez

In [ ]:
!pwd

In [ ]:
%pip install db-dtypes

In [ ]:
import db_dtypes

In [ ]:
%pip install google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pyarrow

In [3]:
import os
import json
import requests
from google.auth import aws
from google.cloud import bigquery
from google.api_core import exceptions as gcp_exc

IMDS = "http://169.254.169.254/latest"

def _imds_token():
    return requests.put(
        f"{IMDS}/api/token",
        headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
        timeout=2,
    ).text

class IMDSv2Supplier(aws.AwsSecurityCredentialsSupplier):
    def get_aws_region(self, context, request):
        az = requests.get(
            f"{IMDS}/meta-data/placement/availability-zone",
            headers={"X-aws-ec2-metadata-token": _imds_token()},
            timeout=2,
        ).text
        return az[:-1]

    def get_aws_security_credentials(self, context, request):
        tok = _imds_token()
        role = requests.get(
            f"{IMDS}/meta-data/iam/security-credentials/",
            headers={"X-aws-ec2-metadata-token": tok},
            timeout=2,
        ).text.strip()
        creds = json.loads(requests.get(
            f"{IMDS}/meta-data/iam/security-credentials/{role}",
            headers={"X-aws-ec2-metadata-token": tok},
            timeout=2,
        ).text)
        return aws.AwsSecurityCredentials(
            access_key_id=creds["AccessKeyId"],
            secret_access_key=creds["SecretAccessKey"],
            session_token=creds.get("Token"),
        )

with open("/home/ubuntu/project/creds.json") as f:
    info = json.load(f)

credentials = aws.Credentials(
    audience=info["audience"],
    subject_token_type=info["subject_token_type"],
    token_url=info["token_url"],
    service_account_impersonation_url=info["service_account_impersonation_url"],
    aws_security_credentials_supplier=IMDSv2Supplier(),
)

client = bigquery.Client(project="project-10a1db63-0045-4848-854", credentials=credentials)

query = "SELECT 'Handshake Successful!' as status"
for row in client.query(query):
    print(row.status)

Handshake Successful!


In [7]:
# Flash Loan Cascade: Deep Sequence Detection
# Goal: Detect complex multi-protocol arbitrage and exploit chains using "No-Join" account arrays.
# Ideal for: Transformer AE "Feature-Attention" and Sequence Profiling.

# ── UPDATED CONFIGURATION FOR CASCADES ────────────────────────────
# This configuration matches the query output below and replaces 
# the old config in the notebook.

DATASET_ID = 'my_dataset_uc3'

# We use the rolling window approach now
names = ['sep']#, 'oct', 'nov', 'dec', 'jan']
chunks = [
    ('2024-09-21', '2024-09-25'),
]

# Removed all 'Instructions' table joins to save $95/month.
# Added Account-Array proxies (unique_program_count, proxy_cpi_depth)
EXPECTED_FEATURE_COLUMNS = [
    # Base Transaction Data
    'signature','block_timestamp','tx_date','wallet','fee_sol','compute_units_consumed',
    'success_flag','num_accounts','log_count','num_balance_changes',
    'max_sol_change',
    
    # Cascade / Protocol Hop Features (The "Cheap" Structural Proxies)
    'watchlist_hop_count', 'unique_program_count', 'proxy_cpi_depth',
    'balance_churn_rate', 'hop_density', 'avg_depth_per_protocol',
    'involves_unknown_protocols_flag',
    
    # Token Transfer Aggregates
    'num_token_transfers', 'mint_diversity', 'total_vol'
]

# This defines the numeric subset passed into the Transformer/Scaler
tx_feature_cols = [
    'fee_sol', 'compute_units_consumed', 'num_accounts', 'log_count', 
    'num_balance_changes', 'max_sol_change', 'watchlist_hop_count', 
    'unique_program_count', 'proxy_cpi_depth', 'balance_churn_rate', 
    'hop_density', 'avg_depth_per_protocol', 'num_token_transfers', 
    'mint_diversity', 'total_vol'
]
# ──────────────────────────────────────────────────────────────────

In [8]:
def get_cascade_query(dataset_id, start_date, end_date, daily_cap=10000):
    limit_clause = f"LIMIT {daily_cap}" if daily_cap else ""
    return f"""
CREATE OR REPLACE TABLE {dataset_id}.flash_loan_cascade_features
PARTITION BY tx_date
CLUSTER BY wallet
AS

WITH cascade_base AS (
  SELECT
    signature,
    block_timestamp,
    DATE(block_timestamp) AS tx_date,
    fee / 1e9 AS fee_sol,
    compute_units_consumed,
    IF(status='Success',1,0) AS success_flag,
    ARRAY_LENGTH(log_messages) AS log_count,
    (SELECT a.pubkey FROM UNNEST(accounts) a WHERE a.signer LIMIT 1) AS wallet,
    
    -- THE WATCHLIST SIGNAL (Known Targets):
    (SELECT COUNT(DISTINCT a.pubkey) 
     FROM UNNEST(accounts) a 
     WHERE a.pubkey IN (
        'JUP6LkbZbjS1jKKwapdHNy74zcZ3tLUZoi5QNyVTaV4', -- Jupiter
        '675kPX9MHTjS2zt1qfr1NYHuzeLXfQM9H24wFSUt1Mp8', -- Raydium
        '6LtLpnUFNByNXLyCoK9wA2MykKAmQNZKBdY8s47dehDc', -- Kamino
        'So1endDq2YkqTCqM3vYpe61CPLUD6nHRN4um8HEufXq', -- Solend
        'whirLuv3caEkYvHWfCHsrTfcTGSW7uRckyAm7pKA5MC', -- Orca Whirlpool
        '6EF8rrecthR5Dkzon8Nwu78hRvfCKubJ14M5uBEwF6P', -- Pump.fun
        'LBUZKhR7JkaNmMbhxuCc3C6rSkmcz6ZXLMa7YjtAn2D', -- Meteora
        'MFv2hWf31Z9kbCa1snEPYpcwiNcNVt7BvR9LpXkXVwN', -- MarginFi
        'dRiftyHA39MWEi3m9aunc5MzRF1JYuBsEX6TUqavNbc'  -- Drift
     )
    ) AS watchlist_hop_count,

    -- THE GENERIC SIGNAL (Zero-Day Discovery):
    -- Count all unique non-system programs to catch unknown protocols in the cascade.
    (SELECT COUNT(DISTINCT a.pubkey) 
     FROM UNNEST(accounts) a 
     WHERE a.pubkey NOT IN (
        'TokenkegQfeZyiNwAJbNbGKPFXCWuBvf9Ss623VQ5DA', -- Token Program
        '11111111111111111111111111111111',           -- System Program
        'ComputeBudget111111111111111111111111111111', -- Compute Budget
        'ATokenGPvbdGVxr1b2hvZbsiqW5xWH25efTNsLJA8knL', -- Assoc Token
        'MemoSq4gqABboxP7eHqJ4znqpZxyHoyWycLCevMfCQC'  -- Memo Program
     )
     AND NOT a.signer
    ) AS unique_program_count,

    -- PROXY FOR CPI DEPTH: Search logs for "invoke" keywords without scanning Instructions table
    (SELECT COUNTIF(REGEXP_CONTAINS(msg, r'Program .* invoke')) FROM UNNEST(log_messages) msg) AS proxy_cpi_depth,
    
    ARRAY_LENGTH(balance_changes) AS balance_churn_rate,
    ARRAY_LENGTH(IFNULL(accounts,[])) AS num_accounts,
    ARRAY_LENGTH(IFNULL(balance_changes,[])) AS num_balance_changes,
    COALESCE((SELECT MAX(ABS(bc.after - bc.before)) FROM UNNEST(balance_changes) bc), 0)/1e9 AS max_sol_change,
    
    -- Extract token features natively from balances! (No-Join)
    ARRAY_LENGTH(IFNULL(pre_token_balances,[])) AS num_token_transfers,
    (SELECT COUNT(DISTINCT tb.mint) FROM UNNEST(pre_token_balances) tb) AS mint_diversity,
    COALESCE((SELECT SUM(CAST(tb.amount AS FLOAT64) / POW(10, NULLIF(tb.decimals, 0))) FROM UNNEST(pre_token_balances) tb), 0) AS total_vol
  FROM `bigquery-public-data.crypto_solana_mainnet_us.Transactions`
  WHERE block_timestamp >= TIMESTAMP('{start_date}') AND block_timestamp < TIMESTAMP_ADD(TIMESTAMP('{end_date}'), INTERVAL 1 DAY)
    -- TRIGGER: Only look at complex transactions touching more than 2 non-system programs
    AND compute_units_consumed > 500000
    -- VALUE FILTER: Ensure it's moving money across multiple tokens
    AND (SELECT COUNT(DISTINCT tb.mint) FROM UNNEST(pre_token_balances) tb) >= 2
    -- WALLET HASH FILTER: Keep exactly 10% of unique wallets to hit ~3M rows total across 5 months
    AND MOD(ABS(FARM_FINGERPRINT((SELECT a.pubkey FROM UNNEST(accounts) a WHERE a.signer LIMIT 1))), 100) < 10
)

SELECT
  c.*,
  -- Engineered features for the Transformer
  SAFE_DIVIDE(c.unique_program_count, NULLIF(c.log_count, 0)) AS hop_density,
  SAFE_DIVIDE(c.proxy_cpi_depth, NULLIF(c.unique_program_count, 0)) AS avg_depth_per_protocol,
  IF(c.unique_program_count > c.watchlist_hop_count, 1, 0) AS involves_unknown_protocols_flag
FROM cascade_base c
WHERE c.unique_program_count >= 2 -- Narrow focus to true Cascades (2+ protocols)
{limit_clause}
"""

In [9]:
# Assuming client is instantiated elsewhere in your real script
# client = bigquery.Client()

def run_validation(client):
    # ---- Pre-live validation + optional live run ----

    # Step 0: ensure dataset exists in us-central1
    dataset_ref = bigquery.DatasetReference(client.project, DATASET_ID)
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = 'us-central1'
    try:
        client.create_dataset(dataset, exists_ok=True)
        print('Dataset ready.')
    except gcp_exc.Forbidden:
        try:
            client.get_dataset(dataset_ref)
            print('Dataset already exists - continuing.')
        except Exception as exc:
            raise RuntimeError(
                f"Dataset '{DATASET_ID}' missing and service account cannot create it. "
                "Create it manually with location us-central1."
            ) from exc
    except gcp_exc.Conflict:
         print('Dataset already exists - continuing.')

    # Step 1: probe source table access
    print("\nProbing access to public Solana transactions table...")
    probe_sql = """
        SELECT COUNT(*) AS n
        FROM `bigquery-public-data.crypto_solana_mainnet_us.Transactions`
        WHERE block_timestamp BETWEEN '2024-09-01' AND '2024-09-01 00:01:00'
    """
    probe_job = client.query(probe_sql, location='us-central1')
    probe = probe_job.result()
    count = next(iter(probe))['n']
    print(f"  Access confirmed - {count:,} rows in first minute of 2024-09-01 (Job ID: {probe_job.job_id})\n")

    # Step 1.5: feature-level sample validation
    print('Running feature-level sample check before live run...')
    sample_start = '2024-09-01'
    sample_end = '2024-09-01'

    # The new query is a single, triggered CTE pipeline, not a two-step random sampler.
    # Therefore, 'presample_pct' is removed. We just increase the daily_cap limit.
    attempts = [
        {'daily_cap': 20},
        {'daily_cap': 100},
        {'daily_cap': 300},
    ]

    sample_q = f"""
        SELECT *
        FROM {DATASET_ID}.flash_loan_cascade_features
        WHERE tx_date BETWEEN DATE('{sample_start}') AND DATE('{sample_end}')
        LIMIT 10
    """

    sample_df = None
    for i, cfg in enumerate(attempts, start=1):
        print(
            f"  Sample attempt {i}/{len(attempts)} "
            f"(daily_cap={cfg['daily_cap']})..."
        )
        
        # Execute the new single-stage Cascade query directly
        build_job = client.query(
            get_cascade_query(
                DATASET_ID,
                sample_start,
                sample_end,
                daily_cap=cfg['daily_cap']
            ),
            location='us-central1',
        )
        build_job.result()
        print(f"    -> Build Job ID: {build_job.job_id}")
        
        extract_job = client.query(sample_q, location='us-central1')
        sample_df = extract_job.to_dataframe()
        print(f"    -> Extract Job ID: {extract_job.job_id}")
        if not sample_df.empty:
            break

    if sample_df is None or sample_df.empty:
        raise RuntimeError(
            'Feature sample query returned 0 rows. '
            'This means no transactions met the Cascade trigger condition (Compute > 500k & 2+ Protocols) on this date. '
            'Try widening sample dates.'
        )

    print(f"Feature sample returned {len(sample_df)} rows.")
    print(f"Feature column count: {len(sample_df.columns)}")
    print('Columns:')
    print(', '.join(sample_df.columns))

    missing = [c for c in EXPECTED_FEATURE_COLUMNS if c not in sample_df.columns]
    if missing:
        raise RuntimeError('Feature schema mismatch. Missing expected columns: ' + ', '.join(missing))

    print('Schema check passed: all expected feature columns are present.')
    print('\nPreview row:')
    print(sample_df.head(1).to_string(index=False))
    print('\n' + '=' * 65 + '\n')

if __name__ == '__main__':
    client = bigquery.Client(project="project-10a1db63-0045-4848-854", credentials=credentials)
    run_validation(client)

Dataset ready.

Probing access to public Solana transactions table...
  Access confirmed - 209,592 rows in first minute of 2024-09-01 (Job ID: adf300ba-a03d-47f2-94f5-236af33b986d)

Running feature-level sample check before live run...
  Sample attempt 1/3 (daily_cap=20)...
    -> Build Job ID: 42373f4c-b082-410c-8e59-225f48e2abcb
    -> Extract Job ID: 96fdc689-b231-4607-9688-fb615e0f3f8e
Feature sample returned 10 rows.
Feature column count: 21
Columns:
signature, block_timestamp, tx_date, fee_sol, compute_units_consumed, success_flag, log_count, wallet, watchlist_hop_count, unique_program_count, proxy_cpi_depth, balance_churn_rate, num_accounts, num_balance_changes, max_sol_change, num_token_transfers, mint_diversity, total_vol, hop_density, avg_depth_per_protocol, involves_unknown_protocols_flag
Schema check passed: all expected feature columns are present.

Preview row:
                                                                               signature           block_timesta

In [10]:
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta

RUN_LIVE = True  # Set to True when you want chunk export.

def downcast_df(df):
    """Reduce RAM usage by downcasting numeric types."""
    fcols = df.select_dtypes('float').columns
    icols = df.select_dtypes('integer').columns
    df[fcols] = df[fcols].apply(pd.to_numeric, downcast='float')
    df[icols] = df[icols].apply(pd.to_numeric, downcast='integer')
    return df

if RUN_LIVE:
    save_dir = '/home/ubuntu/project'
    os.makedirs(save_dir, exist_ok=True)

    for name, (start_str, end_str) in zip(names, chunks):
        start_dt = datetime.strptime(start_str, '%Y-%m-%d')
        end_dt = datetime.strptime(end_str, '%Y-%m-%d')
        
        # Split the month into 5-day sub-chunks to protect RAM
        current_start = start_dt
        chunk_idx = 1
        
        while current_start <= end_dt:
            current_end = min(current_start + timedelta(days=4), end_dt)
            s_str = current_start.strftime('%Y-%m-%d')
            e_str = current_end.strftime('%Y-%m-%d')
            out_path = os.path.join(save_dir, 'sep_test.feather')

            print(f"\n=== {name.upper()} PART {chunk_idx} ({s_str} -> {e_str}) ===")
            
            # 1. Build the features for this sub-chunk
            print('  [1/2] Engineering Cascade features...')
            job1 = client.query(get_cascade_query(DATASET_ID, s_str, e_str, daily_cap=None), location='us-central1')
            job1.result()
            
            # 2. Extract and Downcast
            print('  [2/2] Exporting to feather (with downcasting)...')
            q = f"""
                SELECT *
                FROM {DATASET_ID}.flash_loan_cascade_features
                WHERE tx_date BETWEEN '{s_str}' AND '{e_str}'
            """
            job2 = client.query(q, location='us-central1')
            # to_dataframe() will now use the high-speed Storage API automatically
            df = job2.to_dataframe(progress_bar_type='tqdm')
            
            if not df.empty:
                df = downcast_df(df)
                df.to_feather(out_path)
                
            # Cost Report
            bytes_billed = (job1.total_bytes_billed or 0) + (job2.total_bytes_billed or 0)
            total_gb = bytes_billed / (1024**3)
            print(f"      Rows: {len(df):,}")
            print(f"      Data Scanned: {total_gb:.2f} GB")
            print(f"      Est. Cost: ${ (total_gb/1024) * 5.0:.4f}")
            
            del df # Explicitly free memory
            current_start = current_end + timedelta(days=1)
            chunk_idx += 1
            time.sleep(1)

    print('\nAll chunks complete.')
else:
    print('RUN_LIVE=False, so only pre-live validation was executed.')

[sep Part 1] Skip - exists

All chunks complete.
